In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.gridspec as gridspec
import plotly.express as px
import matplotlib.image as mpimg
import io
plt.rcParams['font.family'] = 'Georgia'

ModuleNotFoundError: No module named 'kaleido'

In [ ]:

df = pd.read_csv("../data/processed/Cleaned_Global_Cybersecurity_Threats_2015-2024.csv")
fig = plt.figure(figsize=(20, 16), constrained_layout=False)
gs = gridspec.GridSpec(4, 6, figure=fig,hspace=0.4, wspace=1)

# Hàng 0
ax1 = fig.add_subplot(gs[0, 0:4])     # gộp 4 cột
ax2 = fig.add_subplot(gs[0, 4])
ax3 = fig.add_subplot(gs[0, 5])

# Hàng 1
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2:4])     # gộp 2 cột
ax7 = fig.add_subplot(gs[1, 4])
ax8 = fig.add_subplot(gs[1, 5])

# Hàng 2
ax9  = fig.add_subplot(gs[2, 0])
ax10 = fig.add_subplot(gs[2, 1])
ax11 = fig.add_subplot(gs[2, 2])
ax12 = fig.add_subplot(gs[2, 3])
ax13 = fig.add_subplot(gs[2, 4:6])    # gộp 2 cột

# Hàng 3
ax14 = fig.add_subplot(gs[3, 0:2])    # gộp 2 cột
ax15 = fig.add_subplot(gs[3, 3:6])    # gộp 3 cột


# Gom lại
axes = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8,
        ax9, ax10, ax11, ax12, ax13, ax14, ax15]

## Name :
ax1.text(0.5, 0.5, "GLOBAL CYBERSECURITY THREATS (2015-2024)",
         fontsize= 28, fontweight='bold', color='#800026',
         ha='center', va='center')
ax1.axis('off') # tắt axis nè 


## financial loss
total_loss = df['Financial Loss (in Million $)'].sum()
ax4.text(0.5, 0.5, f'{ total_loss:,.0f}\nDollars Loss',ha='center', va='center', fontsize=26, fontweight='bold',color = '#800026')
ax4.axis('off') # tắt axis


## afected users
Af_users = df['Number of Affected Users'].sum()
ax5.text(0.5, 0.5, f'{total_loss:,.0f}\nAffected Users',ha='center', va='center', fontsize=26, fontweight='bold',color = '#800026')
ax5.axis('off') # tắt axis


## tỉ lệ phần trăm nguyên nhân của các cuộc tấn công
reasoncount=df['Security Vulnerability Type'].value_counts().sort_values()
labels = reasoncount.index.tolist()
sizes = reasoncount.values
colors = [ '#feb24c', '#ffeda0','#F6DED8','#800026'] 
wedges, texts = ax2.pie(
    sizes,
    labels=labels,  # ẩn nhãn trực tiếp trên lát
    colors=colors,
    startangle=90, 
    pctdistance=0.99,
    wedgeprops={'width': 0.45},
    explode=[0,0,0,0.1],
    textprops={ 'fontsize': 5}
)
ax2.set_title('Percentage of Security Vulnerability \n in cybersecurity attack(%)', fontsize=8, fontweight='bold')
ax2.axis('equal') 


## Number of attack according to Security vulnerability from 2015 to 2024
# Đếm số lượng lỗ hổng theo từng năm
vuln_by_year = df.groupby(['Year', 'Security Vulnerability Type']).size().reset_index(name='Count')
# Pivot để mỗi dòng là năm, mỗi cột là loại lỗ hổng
vuln_pivot = vuln_by_year.pivot(index='Year', columns='Security Vulnerability Type', values='Count').fillna(0)
# Vẽ biểu đồ đường
for col in vuln_pivot.columns:
    if col == 'Zero-Day':
        ax13.plot(vuln_pivot.index, vuln_pivot[col], label=col, color='#800026', linewidth=2.5)
    else:
        ax13.plot(vuln_pivot.index, vuln_pivot[col], label=col,color ='#feb24c', linewidth=1.5)
ax13.set_title('Number of attacks by security vulnerability from 2015 to 2024', fontsize= 8 , fontweight='bold')
ax13.set_xlabel('Year')
ax13.set_ylabel('Amount')


## biểu đồ phòng thủ áp dụng cho từng loại lỗ hổng 
heatmap_data = df.groupby(['Security Vulnerability Type', 'Defense Mechanism Used']).size().unstack(fill_value=0)
# Vẽ heatmap
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap= 'YlOrRd' , linewidths=0.5, ax=ax15,annot_kws={"size": 7})   # cỡ số trong ô)
ax15.set_title('Defense Mechanism used for type of security vulnerability', fontsize=8, fontweight='bold')
ax15.set_xticklabels(ax15.get_xticklabels(), rotation=45)  # Quay nhãn trục x
ax15.set_yticklabels(ax15.get_yticklabels(), rotation=0)   # Quay nhãn trục y


## financial loss theo lỗ hổng
finan_los_type=df[['Financial Loss (in Million $)','Security Vulnerability Type']].groupby('Security Vulnerability Type').sum()
max_loss = finan_los_type['Financial Loss (in Million $)'].idxmax()
filtered_data = df[df['Security Vulnerability Type'] == 'Zero-Day']
numbersecurity=filtered_data[['Security Vulnerability Type','Attack Type']].groupby('Attack Type').count()
# Tạo danh sách màu: nếu index là cột cao nhất thì dùng màu khác, còn lại dùng màu mặc định
colors = ['#800026' if i == max_loss else'#ffeda0' for i in finan_los_type.index]
bars = ax8.bar(finan_los_type.index, finan_los_type['Financial Loss (in Million $)'], color=colors)
ax8.set_title('Cybersecurity Loss According to \n Security Vulnerability Type (2015-2024)', fontsize=8, fontweight='bold')
ax8.set_xticklabels(finan_los_type.index, rotation=45, fontsize=4)
max_index = numbersecurity['Security Vulnerability Type'].idxmax()
colors = ['#800026' if i == max_index else '#ffeda0' for i in numbersecurity.index]


## Number of attack caused by security vulnerability:Zero-Day according to Attack Type
filtered_data = df[df['Security Vulnerability Type'] == 'Zero-Day']
numbersecurity=filtered_data[['Security Vulnerability Type','Attack Type']].groupby('Attack Type').count()
bars = ax7.bar(numbersecurity.index, numbersecurity['Security Vulnerability Type'], color=colors)
ax7.set_title ( 'Number of attack to security vulnerability:\nZero-day according to Attack Type', fontsize=8, fontweight='bold')
ax7.set_ylabel('Number attack to Zero-day vulneralbility')
ax7.set_xticklabels(numbersecurity.index, rotation=45, fontsize=4)
max_value = numbersecurity['Security Vulnerability Type'].max()
max_x = list(numbersecurity.index).index(max_index)  # vị trí theo thứ tự cột

# Hiển thị giá trị ngay trên đỉnh cột cao nhất
ax7.text(max_x, max_value + 0.5, f'{max_value}', ha='center', va='bottom', fontsize=10,color='#800026')
attack_counts = filtered_data['Attack Type'].value_counts().sort_values(ascending=False)


## %Phishing và Other
phishing_count = attack_counts.get('Phishing', 0)
other_count = attack_counts.sum() - phishing_count
labels = ['Other', 'Phishing ']
sizes = [other_count, phishing_count]
colors = ['#feb24c', '#800026']  # màu đỏ đậm cho Phishing, đỏ nhạt cho phần còn lại
wedges, texts= ax3.pie(
    sizes,
    labels=labels,
    colors=colors,
    explode=[0,0.15],
    startangle=90,
    textprops={'color': 'black', 'fontsize': 10},
)
# Tiêu đề
ax3.set_title('Percentage of attack caused by Phishing to:\n Zero-Day vulneralibity',
             fontsize=8,fontweight='bold')
ax3.axis('equal')  # đảm bảo hình tròn



## atttack through year
attacks_per_year = df['Year'].value_counts().sort_index()
ax10.bar(attacks_per_year.index, attacks_per_year.values, color='#800026')
ax10.set_xlabel("Year", fontsize=8)
ax10.set_ylabel("Number of Attacks", fontsize=8)
ax10.set_title("Number of Cyber Attacks Over the Years", fontsize=8, fontweight='bold')
ax10.set_xticklabels(attacks_per_year.index, rotation=0, fontsize=4)


## Financial Loss (in Million $) and Number of Affected Users
grouped_data = df.groupby('Year')[['Financial Loss (in Million $)', 'Number of Affected Users']].sum()
ax6.bar(grouped_data.index, grouped_data['Financial Loss (in Million $)'], color='#feb24c', label='Financial Loss (in Million $)', width=0.5, alpha=0.7)
ax6.set_xlabel('Year', fontsize=8)
ax6.tick_params(axis='y', labelcolor='#feb24c', labelsize=8)
ax6.tick_params(axis='y', labelcolor='#feb24c', labelsize =8)
axx = ax6.twinx()
axx.plot(grouped_data.index, grouped_data['Number of Affected Users'], color='#800026', label='Number of Affected Users')
axx.set_ylabel('Number of Affected Users', color='#800026', fontsize=8)
axx.tick_params(axis='y', labelcolor='#800026', labelsize=8)
ax6.set_title('Financial Loss and Number of Affected Users Over the Years (2015-2024)', fontsize=8, fontweight='bold')

## Thiệt hại phân theo ngành 
industry_loss = df.groupby('Target Industry')['Financial Loss (in Million $)'].sum().reset_index()
industry_loss = industry_loss.sort_values(by='Financial Loss (in Million $)', ascending=False)
max_loss_index = industry_loss['Financial Loss (in Million $)'].idxmax()
colors = ['#800026']
ax11.barh(industry_loss['Target Industry'], industry_loss['Financial Loss (in Million $)'], color=colors)
ax11.set_xlabel('Financial Loss (in Million $)', fontsize=4)
ax11.set_ylabel('Target Industry', fontsize=4)
ax11.set_title('Financial Loss by Target Industry', fontsize=8, fontweight='bold')
ax11.invert_yaxis()  

## % Hình thức tấn công
attack_counts = df['Attack Type'].value_counts()
colors=['#f03b20', '#feb24c', '#ffeda0', '#F6DED8', '#d7301f','#fee08b']
ax12.pie(attack_counts, labels=attack_counts.index, autopct='%1.1f%%', startangle=140, colors = colors )
ax12.set_title('The percentage distribution of \n cybersecurity attack types (2015–2024)', fontsize=9, fontweight='bold')


## world map
country_attacks = df['Country'].str.strip().value_counts().reset_index()
country_attacks.columns = ['Country', 'Attack count']
fig = px.choropleth(
    country_attacks,
    locations='Country',
    locationmode='country names',
    color='Attack count',
    color_continuous_scale='ylorrd',
    title='Map of Most Attacked Countries'
)
fig.update_layout(margin={"l": 20, "r": 20, "t": 40, "b": 20})
img_bytes = fig.to_image(format="png", width=2400, height=1350, engine="kaleido")
img = mpimg.imread(io.BytesIO(img_bytes), format='png')
# chuyển về dạng ảnh png vì plotly không hỗ trợ điền trực típ với matplotlib 
# cài đặt pip install -U kaleido
ax14.imshow(img)
ax14.axis('off')
ax14.set_title('Map of Most Attacked Countries', fontsize=10, fontweight='bold')


## crime index
data = {
    "Crime Type": ["Flora crimes", "Heroin trade", "Cyber-dependent crimes", "Extortion and protection racketeering", "Mafia-style groups","Government transparency and accountability","Victim and witness support"],
    "Index": [4.06,4.08,4.55,4.02,4.02,4.36,4.24]
}
df = pd.DataFrame(data)
# Vẽ biểu đồ thanh ngang
ax9.barh(df["Crime Type"], df["Index"], color=["#800026" if crime == "Cyber-dependent crimes" else "#F6DED8" 
                                                for crime in df["Crime Type"]],)
 # barh = horizontal bar
ax9.set_title("Global crime type index in 2023", fontsize=10, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
ax9.set_yticklabels([]) 

plt.show()



C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14016\3060898909.py:107: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax8.set_xticklabels(finan_los_type.index, rotation=45, fontsize=4)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14016\3060898909.py:118: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax7.set_xticklabels(numbersecurity.index, rotation=45, fontsize=4)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14016\3060898909.py:154: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax10.set_xticklabels(attacks_per_year.index, rotation=0, fontsize=4)
